## Data Cleaning

In [3]:
import pandas as pd
import re

def clean_full_priceoye_dataset(input_csv="large data.csv", output_csv="cleaned_dataset_rag.csv"):
    df = pd.read_csv(input_csv)
    
    # 1. Consolidate Color Variants (Names, Statuses, Image URLs)
    variant_indices = [int(col.split("/")[1]) for col in df.columns if col.startswith("color_variants/") and col.split("/")[1].isdigit()]
    max_idx = max(variant_indices) if variant_indices else 10
    
    c_names, c_statuses, c_images = [], [], []
    for _, row in df.iterrows():
        n, s, img = [], [], []
        for i in range(max_idx + 1):
            n_col, s_col, img_col = f"color_variants/{i}/name", f"color_variants/{i}/status", f"color_variants/{i}/image"
            if n_col in row and pd.notna(row[n_col]) and str(row[n_col]).strip() != "":
                n.append(str(row[n_col]).strip())
                s.append(str(row[s_col]).strip() if (s_col in row and pd.notna(row[s_col])) else "N/A")
                img.append(str(row[img_col]).strip() if (img_col in row and pd.notna(row[img_col])) else "N/A")
        c_names.append(" | ".join(n))
        c_statuses.append(" | ".join(s))
        c_images.append(" | ".join(img))
        
    df["color_names"] = c_names
    df["color_statuses"] = c_statuses
    df["color_images"] = c_images
    df.drop(columns=[c for c in df.columns if c.startswith("color_variants/")], inplace=True)

    # 2. Consolidate What's In The Box
    box_cols = [c for c in df.columns if c.startswith("whats_in_the_box/")]
    if box_cols:
        df["whats_in_the_box"] = df[box_cols].apply(
            lambda r: ", ".join([str(v).strip() for v in r if pd.notna(v) and str(v).strip() != ""]), axis=1
        )
        df.drop(columns=box_cols, inplace=True)

    # 3. Consolidate Delivery Badges
    badge_cols = [c for c in df.columns if c.startswith("warranty_delivery/badges/")]
    if badge_cols:
        df["delivery_badges"] = df[badge_cols].apply(
            lambda r: " | ".join([str(v).strip() for v in r if pd.notna(v) and str(v).strip() != ""]), axis=1
        )
        df.drop(columns=badge_cols, inplace=True)

    # 4. Consolidate FAQs into Single Text Block
    faq_cols = [c for c in df.columns if c.startswith("faqs/")]
    if faq_cols:
        faq_texts = []
        for _, row in df.iterrows():
            faqs = []
            for i in range(10):
                q_col, a_col = f"faqs/{i}/question", f"faqs/{i}/answer"
                if q_col in row and pd.notna(row[q_col]):
                    q = str(row[q_col]).strip()
                    a = str(row[a_col]).strip() if (a_col in row and pd.notna(row[a_col])) else ""
                    faqs.append(f"Q: {q} A: {a}")
            faq_texts.append(" | ".join(faqs))
        df["faqs_text"] = faq_texts
        df.drop(columns=faq_cols, inplace=True)

    # 5. Consolidate User Reviews into Single Text Block
    review_cols = [c for c in df.columns if c.startswith("reviews/")]
    if review_cols:
        review_texts = []
        for _, row in df.iterrows():
            revs = []
            for i in range(10):
                u_col, r_col, c_col = f"reviews/{i}/user", f"reviews/{i}/rating", f"reviews/{i}/comment"
                if c_col in row and pd.notna(row[c_col]):
                    user = str(row[u_col]).strip() if (u_col in row and pd.notna(row[u_col])) else "Anonymous"
                    rating = str(row[r_col]).strip() if (r_col in row and pd.notna(row[r_col])) else ""
                    comment = str(row[c_col]).strip()
                    revs.append(f"{user} ({rating}): {comment}")
            review_texts.append(" | ".join(revs))
        df["reviews_text"] = review_texts
        df.drop(columns=review_cols, inplace=True)

    # Helper function for numeric conversion
    def extract_digits(series):
        return pd.to_numeric(
            series.fillna("0").astype(str).str.replace(r"\D", "", regex=True).replace("", "0"),
            errors="coerce"
        ).fillna(0).astype(int)

    # 6. Extract Pricing Numerics
    if "pricing/current_price" in df.columns:
        df["price_numeric"] = extract_digits(df["pricing/current_price"])
    if "pricing/original_price" in df.columns:
        df["original_price_numeric"] = extract_digits(df["pricing/original_price"])
    if "pricing/discount" in df.columns:
        df["discount_pct"] = extract_digits(df["pricing/discount"])

    # 7. Extract Numeric Specs (Battery, RAM, Storage, Camera, Display)
    if "specs/battery" in df.columns:
        df["battery_mah"] = (
            df["specs/battery"]
            .astype(str)
            .str.extract(r"(\d{4,5})\s*mAh", flags=re.IGNORECASE, expand=False)
            .fillna(df["specs/battery"].astype(str).str.extract(r"(\d{4,5})", expand=False))
            .fillna("0")
            .astype(int)
        )

    if "specs/memory" in df.columns:
        df["ram_gb"] = (
            df["specs/memory"]
            .astype(str)
            .str.extract(r"(\d{1,2})\s*GB\s*RAM", flags=re.IGNORECASE, expand=False)
            .fillna("0")
            .astype(int)
        )
        df["storage_gb"] = (
            df["specs/memory"]
            .astype(str)
            .str.extract(r"(\d{2,4})\s*GB(?!\s*RAM)", flags=re.IGNORECASE, expand=False)
            .fillna("0")
            .astype(int)
        )

    if "specs/display" in df.columns:
        df["display_size_inches"] = (
            pd.to_numeric(
                df["specs/display"]
                .astype(str)
                .str.extract(r"(\d{1,2}\.\d{1,2})\s*inch", flags=re.IGNORECASE, expand=False),
                errors="coerce"
            ).fillna(0.0)
        )

    if "specs/camera" in df.columns:
        df["main_camera_mp"] = (
            df["specs/camera"]
            .astype(str)
            .str.extract(r"(\d{1,3})\s*MP", flags=re.IGNORECASE, expand=False)
            .fillna("0")
            .astype(int)
        )

    # 8. Clean up Column Names (Remove Slashes -> snake_case)
    rename_mapping = {
        "pricing/current_price": "current_price_str",
        "pricing/original_price": "original_price_str",
        "pricing/discount": "discount_str",
        "pricing/pta_status": "pta_status",
        "warranty_delivery/warranty_terms": "warranty_terms",
        "specs/display": "display",
        "specs/memory": "memory",
        "specs/processor": "processor",
        "specs/battery": "battery",
        "specs/camera": "camera",
    }
    df.rename(columns=rename_mapping, inplace=True)

    # Save finalized dataset
    df.to_csv(output_csv, index=False)
    print(f"Dataset successfully cleaned and saved to '{output_csv}'!")

# Run cleaning process
clean_full_priceoye_dataset("large data.csv", "cleaned_dataset_rag.csv")

Dataset successfully cleaned and saved to 'cleaned_dataset_rag.csv'!
